Adaptive Diabetes Risk Prediction with Model Self-Improvement Under Data Change

In [1]:
import numpy as np
import pandas as pd

from sklearn.model_selection import train_test_split, StratifiedKFold, GridSearchCV
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, roc_auc_score, confusion_matrix
from sklearn.cluster import KMeans

from scipy.stats import ks_2samp


In [2]:
RANDOM_STATE = 42
N_CLUSTERS = 4
CV_SPLITS = 5
SCORING = "f1"
EPS = 1e-6
DRIFT_PVAL_THRESHOLD = 0.01

def evaluate(model, X, y, title="Evaluation"):
    y_pred = model.predict(X)
    y_proba = model.predict_proba(X)[:, 1]
    print(f"\n=== {title} ===")
    print(classification_report(y, y_pred, digits=3))
    print("ROC-AUC:", round(roc_auc_score(y, y_proba), 3))
    print("Confusion matrix:\n", confusion_matrix(y, y_pred))

def ks_drift(old_df, new_df, cols, p_thresh=DRIFT_PVAL_THRESHOLD):
    for c in cols:
        if c in old_df.columns and c in new_df.columns:
            a = pd.to_numeric(old_df[c], errors="coerce")
            b = pd.to_numeric(new_df[c], errors="coerce")
            a = a.replace([np.inf, -np.inf], np.nan).dropna()
            b = b.replace([np.inf, -np.inf], np.nan).dropna()
            if len(a) > 30 and len(b) > 30:
                _, p = ks_2samp(a, b)
                if p < p_thresh:
                    return True
    return False


In [4]:
df = pd.read_csv('/content/diabetes_1 (1).csv')

X_full = df.drop(columns=["Outcome"]).copy()
y_full = df["Outcome"].astype(int).copy()

numeric_cols = X_full.columns.tolist()


In [5]:
print(df.isnull().sum())

Pregnancies                 0
Glucose                     0
BloodPressure               0
SkinThickness               0
Insulin                     0
BMI                         0
DiabetesPedigreeFunction    0
Age                         0
Outcome                     0
dtype: int64


In [6]:
scaler = StandardScaler()
logreg = LogisticRegression(
    max_iter=500,
    solver="liblinear",
    random_state=RANDOM_STATE
)

pipe_baseline = Pipeline([
    ("scaler", scaler),
    ("clf", logreg)
])


In [7]:
param_grid = {
    "clf__penalty": ["l1", "l2"],
    "clf__C": [0.01, 0.1, 1, 3, 10]
}


In [8]:
X_train_base, X_test_base, y_train_base, y_test_base = train_test_split(
    X_full, y_full, test_size=0.2, stratify=y_full, random_state=RANDOM_STATE
)

cv = StratifiedKFold(n_splits=CV_SPLITS, shuffle=True, random_state=RANDOM_STATE)

gs_base = GridSearchCV(
    estimator=pipe_baseline,
    param_grid=param_grid,
    cv=cv,
    scoring=SCORING,
    n_jobs=-1
)
gs_base.fit(X_train_base, y_train_base)

baseline_model = gs_base.best_estimator_
print("Baseline best params:", gs_base.best_params_)
evaluate(
    baseline_model,
    X_test_base,
    y_test_base,
    title="Baseline (no tailoring, no clusters)"
)


Baseline best params: {'clf__C': 0.1, 'clf__penalty': 'l1'}

=== Baseline (no tailoring, no clusters) ===
              precision    recall  f1-score   support

           0      0.743     0.810     0.775       100
           1      0.578     0.481     0.525        54

    accuracy                          0.695       154
   macro avg      0.660     0.646     0.650       154
weighted avg      0.685     0.695     0.688       154

ROC-AUC: 0.812
Confusion matrix:
 [[81 19]
 [28 26]]


In [9]:
X_tailored = X_full.copy()

if {"Insulin", "Glucose"}.issubset(X_tailored.columns):
    X_tailored["Insulin_over_Glucose"] = (
        X_tailored["Insulin"] / (X_tailored["Glucose"] + EPS)
    )

if {"BMI", "Glucose"}.issubset(X_tailored.columns):
    X_tailored["BMI_x_Glucose"] = (
        X_tailored["BMI"] * X_tailored["Glucose"]
    )


In [10]:
X_train_tailored, X_test_tailored, y_train_tailored, y_test_tailored = train_test_split(
    X_tailored, y_full, test_size=0.2, stratify=y_full, random_state=RANDOM_STATE
)

pipe_tailored = Pipeline([
    ("scaler", StandardScaler()),
    ("clf", LogisticRegression(max_iter=500, solver="liblinear", random_state=RANDOM_STATE))
])

gs_tailored = GridSearchCV(
    estimator=pipe_tailored,
    param_grid=param_grid,
    cv=cv,
    scoring=SCORING,
    n_jobs=-1
)
gs_tailored.fit(X_train_tailored, y_train_tailored)

tailored_model = gs_tailored.best_estimator_
print("\nData Tailoring best params:", gs_tailored.best_params_)
evaluate(
    tailored_model,
    X_test_tailored,
    y_test_tailored,
    title="After Data Tailoring (engineered features added)"
)



Data Tailoring best params: {'clf__C': 0.01, 'clf__penalty': 'l2'}

=== After Data Tailoring (engineered features added) ===
              precision    recall  f1-score   support

           0      0.784     0.800     0.792       100
           1      0.615     0.593     0.604        54

    accuracy                          0.727       154
   macro avg      0.700     0.696     0.698       154
weighted avg      0.725     0.727     0.726       154

ROC-AUC: 0.813
Confusion matrix:
 [[80 20]
 [22 32]]


In [11]:
scaler_for_kmeans = StandardScaler().fit(X_train_tailored)
Z_train_tailored = scaler_for_kmeans.transform(X_train_tailored)

kmeans = KMeans(
    n_clusters=N_CLUSTERS,
    n_init=10,
    random_state=RANDOM_STATE
).fit(Z_train_tailored)

def add_cluster_feature(X, fitted_scaler, fitted_kmeans):
    Z = fitted_scaler.transform(X)
    clusters = fitted_kmeans.predict(Z)
    Xc = X.copy()
    Xc["cluster_id"] = clusters.astype("float64")
    return Xc

X_train_clustered = add_cluster_feature(
    X_train_tailored, scaler_for_kmeans, kmeans
)
X_test_clustered = add_cluster_feature(
    X_test_tailored, scaler_for_kmeans, kmeans
)

pipe_clustered = Pipeline([
    ("scaler", StandardScaler()),
    ("clf", LogisticRegression(max_iter=500, solver="liblinear", random_state=RANDOM_STATE))
])

gs_clustered = GridSearchCV(
    estimator=pipe_clustered,
    param_grid=param_grid,
    cv=cv,
    scoring=SCORING,
    n_jobs=-1
)
gs_clustered.fit(X_train_clustered, y_train_tailored)

clustered_model = gs_clustered.best_estimator_
print("\nClustering+Prediction best params:", gs_clustered.best_params_)
evaluate(
    clustered_model,
    X_test_clustered,
    y_test_tailored,
    title="After Clustering + Prediction (cluster_id added)"
)



Clustering+Prediction best params: {'clf__C': 0.1, 'clf__penalty': 'l2'}

=== After Clustering + Prediction (cluster_id added) ===
              precision    recall  f1-score   support

           0      0.752     0.790     0.771       100
           1      0.571     0.519     0.544        54

    accuracy                          0.695       154
   macro avg      0.662     0.654     0.657       154
weighted avg      0.689     0.695     0.691       154

ROC-AUC: 0.817
Confusion matrix:
 [[79 21]
 [26 28]]


In [12]:
if "Age" in X_tailored.columns:
    mask_young = X_tailored["Age"] < 40
else:
    mask_young = X_tailored["Glucose"] < X_tailored["Glucose"].median()

X_cohort_young = X_tailored[mask_young].copy()
y_cohort_young = y_full[mask_young].copy()
X_cohort_older = X_tailored[~mask_young].copy()
y_cohort_older = y_full[~mask_young].copy()


In [13]:
X_train_initial, X_valid_initial, y_train_initial, y_valid_initial = train_test_split(  # Train/valid within YoungAdults
    X_cohort_young, y_cohort_young, test_size=0.2, stratify=y_cohort_young, random_state=RANDOM_STATE
)

In [14]:
scaler_initial_for_kmeans = StandardScaler().fit(X_train_initial)
Z_train_initial = scaler_initial_for_kmeans.transform(X_train_initial)

kmeans_initial = KMeans(
    n_clusters=N_CLUSTERS, n_init=10, random_state=RANDOM_STATE
).fit(Z_train_initial)

def add_cluster_initial(X):
    Z = scaler_initial_for_kmeans.transform(X)
    cl = kmeans_initial.predict(Z)
    Xc = X.copy()
    Xc["cluster_id"] = cl.astype("float64")
    return Xc

X_train_initial_c = add_cluster_initial(X_train_initial)
X_valid_initial_c = add_cluster_initial(X_valid_initial)

pipe_initial = Pipeline([
    ("scaler", StandardScaler()),
    ("clf", LogisticRegression(max_iter=500, solver="liblinear", random_state=RANDOM_STATE))
])

gs_initial = GridSearchCV(
    estimator=pipe_initial,
    param_grid=param_grid,
    cv=cv,
    scoring=SCORING,
    n_jobs=-1
)
gs_initial.fit(X_train_initial_c, y_train_initial)

initial_model = gs_initial.best_estimator_
print("\nInitial Cohort best params:", gs_initial.best_params_)
evaluate(
    initial_model,
    X_valid_initial_c,
    y_valid_initial,
    title="Model on Initial Cohort (YoungAdults)"
)



Initial Cohort best params: {'clf__C': 0.1, 'clf__penalty': 'l2'}

=== Model on Initial Cohort (YoungAdults) ===
              precision    recall  f1-score   support

           0      0.844     0.938     0.889        81
           1      0.783     0.562     0.655        32

    accuracy                          0.832       113
   macro avg      0.814     0.750     0.772       113
weighted avg      0.827     0.832     0.823       113

ROC-AUC: 0.895
Confusion matrix:
 [[76  5]
 [14 18]]


In [15]:
drift_detected = ks_drift(
    X_train_initial,
    X_cohort_older,
    cols=list(set(X_train_initial.columns) & set(X_cohort_older.columns))
)
print("\nDistribution drift vs. Initial Cohort detected?", drift_detected)



Distribution drift vs. Initial Cohort detected? True


In [16]:
X_combined = pd.concat([X_train_initial, X_cohort_older], axis=0)
y_combined = pd.concat([y_train_initial, y_cohort_older], axis=0)

scaler_combined_for_kmeans = StandardScaler().fit(X_combined)
Z_combined = scaler_combined_for_kmeans.transform(X_combined)

kmeans_combined = KMeans(
    n_clusters=N_CLUSTERS, n_init=10, random_state=RANDOM_STATE
).fit(Z_combined)

def add_cluster_combined(X):
    Z = scaler_combined_for_kmeans.transform(X)
    cid = kmeans_combined.predict(Z)
    Xc = X.copy()
    Xc["cluster_id"] = cid.astype("float64")
    return Xc

X_combined_c = add_cluster_combined(X_combined)
X_older_c = add_cluster_combined(X_cohort_older)

pipe_adapted = Pipeline([
    ("scaler", StandardScaler()),
    ("clf", LogisticRegression(max_iter=500, solver="liblinear", random_state=RANDOM_STATE))
])

gs_adapted = GridSearchCV(
    estimator=pipe_adapted,
    param_grid=param_grid,
    cv=cv,
    scoring=SCORING,
    n_jobs=-1
)
gs_adapted.fit(X_combined_c, y_combined)

adapted_model = gs_adapted.best_estimator_
print("Adapted Model (Combined Cohorts) best params:", gs_adapted.best_params_)
evaluate(
    adapted_model,
    X_older_c,
    y_cohort_older,
    title="After Adaptation (evaluated on OlderAdults)"
)


Adapted Model (Combined Cohorts) best params: {'clf__C': 3, 'clf__penalty': 'l2'}

=== After Adaptation (evaluated on OlderAdults) ===
              precision    recall  f1-score   support

           0      0.667     0.727     0.696        99
           1      0.727     0.667     0.696       108

    accuracy                          0.696       207
   macro avg      0.697     0.697     0.696       207
weighted avg      0.698     0.696     0.696       207

ROC-AUC: 0.747
Confusion matrix:
 [[72 27]
 [36 72]]
